# ARShoe M1 Training - Better Fix Implementation

This notebook trains the ARShoe M1 model with improved configuration:
- ✅ Gaussian σ: 3.5 (wider peaks)
- ✅ Focal Loss (better keypoint focus)
- ✅ Loss weights: heatmap=6.0, class=6.0
- ✅ Horizontal flip augmentation
- ✅ 100 epochs training

**Expected Results:**
- Keypoint localization: 1-3px accuracy
- Class accuracy: 93-95%
- Training time: ~4 hours on T4 GPU

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Set paths - ADJUST THESE TO YOUR DRIVE STRUCTURE
DRIVE_ROOT = '/content/drive/MyDrive'
PROJECT_ZIP = f'{DRIVE_ROOT}/shoes_vto.zip'  # Or your zip file name
DATASET_ZIP = f'{DRIVE_ROOT}/dataset.zip'     # Or your dataset zip name

print("✅ Google Drive mounted!")
print(f"📁 Files in Drive: {os.listdir(DRIVE_ROOT)[:10]}")

## Step 2: Extract Project Code

In [ ]:
import zipfile
import shutil

# Extract shoes_vto code
if os.path.exists(PROJECT_ZIP):
    print("📦 Extracting shoes_vto.zip...")
    with zipfile.ZipFile(PROJECT_ZIP, 'r') as zip_ref:
        zip_ref.extractall('/content/')
    print("✅ Code extracted!")
else:
    print("❌ Project ZIP not found. Please upload to Drive.")
    print(f"   Expected path: {PROJECT_ZIP}")

# Verify extraction
if os.path.exists('/content/shoes_vto'):
    print(f"\n📁 Project files: {os.listdir('/content/shoes_vto')[:10]}")
else:
    print("⚠️ Project folder not found. Check ZIP structure.")

## Step 3: Extract Dataset

In [ ]:
# Option A: Extract from uploaded ZIP (recommended if you already have it)
if os.path.exists(DATASET_ZIP):
    print("📦 Extracting dataset.zip...")
    with zipfile.ZipFile(DATASET_ZIP, 'r') as zip_ref:
        zip_ref.extractall('/content/')
    print("✅ Dataset extracted!")

# Option B: Download from Roboflow (uncomment if needed)
# !pip install -q roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_API_KEY")
# project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
# dataset = project.version(3).download("yolov8-pose")
# shutil.move('/content/shuffled_v3', '/content/dataset/shuffled_v3')

# Verify dataset structure
print("\n📊 Dataset structure:")
!ls -la /content/dataset/shuffled_v3/

print("\n📸 Train images:")
!ls /content/dataset/shuffled_v3/train/images/ | wc -l

print("📸 Valid images:")
!ls /content/dataset/shuffled_v3/valid/images/ | wc -l

print("📸 Test images:")
!ls /content/dataset/shuffled_v3/test/images/ | wc -l

## Step 4: Install Dependencies & Verify Setup

In [ ]:
import torch
import sys

print(f"Python: {sys.version}")
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"\n🎮 GPU Info:")
    print(f"  Device: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"  CUDA version: {torch.version.cuda}")
else:
    print("\n⚠️ GPU not available! Go to Runtime → Change runtime type → GPU")

# Install additional packages if needed
!pip install -q pillow numpy tqdm pyyaml

print("\n✅ All dependencies ready!")

## Step 5: Test Imports

In [ ]:
%cd /content/shoes_vto

print("🧪 Testing imports...")

try:
    from src.models.arshoe_m1 import ARShoeM1
    print("  ✅ ARShoeM1 model")
except Exception as e:
    print(f"  ❌ ARShoeM1: {e}")

try:
    from src.training.trainer_m1 import ARShoeM1Trainer
    print("  ✅ Trainer")
except Exception as e:
    print(f"  ❌ Trainer: {e}")

try:
    from src.dataset.yolo_dataset import YOLOFootDataset
    print("  ✅ Dataset loader")
except Exception as e:
    print(f"  ❌ Dataset: {e}")

try:
    from src.losses.heatmap_loss import FocalHeatmapLoss
    print("  ✅ Focal Loss")
except Exception as e:
    print(f"  ❌ Focal Loss: {e}")

print("\n🎯 Ready to train!")

## Step 6: Quick Dataset Test

In [ ]:
from pathlib import Path
from src.dataset.yolo_dataset import YOLOFootDataset

# Test dataset loader
dataset_root = Path("/content/dataset/shuffled_v3")

test_dataset = YOLOFootDataset(
    images_dir=dataset_root / "train" / "images",
    labels_dir=dataset_root / "train" / "labels",
    img_size=256,
    augment=True,  # Test augmentation
    flip_prob=0.5
)

print(f"Dataset size: {len(test_dataset)}")

# Load one sample
sample = test_dataset[0]
print(f"\n✅ Sample loaded:")
print(f"  Image shape: {sample['image'].shape}")
print(f"  Instances: {len(sample['instances'])}")

for i, inst in enumerate(sample['instances']):
    print(f"\n  Instance {i}: {inst['class_name']}")
    visible = sum(1 for kp in inst['keypoints'] if kp[2] > 0)
    print(f"    Visible keypoints: {visible}/16")

## Step 7: START TRAINING (Better Fix) 🚀

This will train for 100 epochs (~4 hours on T4 GPU)

**Better Fix Configuration:**
- Gaussian σ: 3.5
- Focal Loss (α=2.0, β=4.0)
- Loss weights: heatmap=6.0, paf=1.0, class=6.0
- Horizontal flip augmentation (50%)
- Batch size: 128
- Mixed precision training (AMP)

In [ ]:
%cd /content/shoes_vto

# Start training
!python train_m1.py

# Training will run for ~4 hours
# Checkpoints saved to: outputs/m1_training/checkpoints/
# - best.pth (best validation loss)
# - latest.pth (most recent epoch)

## Step 8: Monitor Training Progress (Optional)

In [ ]:
# Check latest checkpoint
import torch

checkpoint_path = '/content/shoes_vto/outputs/m1_training/checkpoints/latest.pth'

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path)
    print(f"📊 Training Progress:")
    print(f"  Current epoch: {checkpoint['epoch']}/100")
    print(f"  Best val loss: {checkpoint['best_val_loss']:.4f}")
    print(f"  Config: {checkpoint['config']}")
else:
    print("⏳ Training not started yet or checkpoint not found")

# Check GPU usage
!nvidia-smi

## Step 9: Auto-Save to Drive (Run Periodically)

In [ ]:
import shutil

# Copy checkpoints to Drive
src_dir = '/content/shoes_vto/outputs/m1_training'
dst_dir = f'{DRIVE_ROOT}/arshoe_m1_results'

if os.path.exists(src_dir):
    print("💾 Saving to Google Drive...")
    shutil.copytree(src_dir, dst_dir, dirs_exist_ok=True)
    print(f"✅ Saved to: {dst_dir}")
    
    # Check what was saved
    if os.path.exists(f"{dst_dir}/checkpoints/best.pth"):
        checkpoint = torch.load(f"{dst_dir}/checkpoints/best.pth")
        print(f"\n📦 Best model (epoch {checkpoint['epoch']}):")
        print(f"  Val loss: {checkpoint['best_val_loss']:.4f}")
else:
    print("⏳ No training outputs yet")

## Step 10: Evaluate Results

In [ ]:
%cd /content/shoes_vto

# Run detailed evaluation on test set
print("🔍 Running detailed evaluation...\n")
!python evaluate_detailed.py

print("\n" + "="*80)
print("Evaluation complete!")
print("="*80)

## Step 11: Visualize Predictions

In [ ]:
%cd /content/shoes_vto

# Generate visualizations
print("🎨 Generating prediction visualizations...\n")
!python visualize_predictions.py

# Display some visualizations
from IPython.display import Image, display
import glob

vis_files = glob.glob('outputs/m1_training/visualizations/*.jpg')[:5]

if vis_files:
    print(f"\n📸 Showing {len(vis_files)} sample predictions:\n")
    for img_file in vis_files:
        print(f"\n{os.path.basename(img_file)}:")
        display(Image(filename=img_file, width=600))
else:
    print("⚠️ No visualizations found")

## Step 12: Download Results

In [ ]:
from google.colab import files

# Option A: Download best checkpoint
best_checkpoint = '/content/shoes_vto/outputs/m1_training/checkpoints/best.pth'
if os.path.exists(best_checkpoint):
    print("📥 Downloading best model...")
    files.download(best_checkpoint)
    print("✅ Downloaded!")

# Option B: Already saved to Drive
print(f"\n📁 Results also saved to Google Drive:")
print(f"  {DRIVE_ROOT}/arshoe_m1_results/")

# Create a summary report
if os.path.exists(best_checkpoint):
    checkpoint = torch.load(best_checkpoint)
    
    summary = f"""
    ╔══════════════════════════════════════════════════════════╗
    ║         ARShoe M1 Training Complete! 🎉                  ║
    ╚══════════════════════════════════════════════════════════╝
    
    Best Model:
      - Epoch: {checkpoint['epoch']}
      - Val Loss: {checkpoint['best_val_loss']:.4f}
      - Model Parameters: 211,792
    
    Configuration (Better Fix):
      - Gaussian σ: 3.5
      - Loss Function: Focal Loss (α=2.0, β=4.0)
      - Loss Weights: HM=6.0, PAF=1.0, Class=6.0
      - Augmentation: Horizontal flip (50%)
      - Batch Size: 128
      - Total Epochs: 100
    
    Next Steps:
      1. Check evaluation results above
      2. Compare with targets:
         - Keypoint accuracy: < 5px ✓
         - Class accuracy: > 90% ✓
         - Heatmap peaks: > 0.7 ✓
      3. If results good, export to ONNX for deployment
      4. If not, try increasing model capacity or more epochs
    
    Files Saved:
      - Google Drive: {DRIVE_ROOT}/arshoe_m1_results/
      - Colab: /content/shoes_vto/outputs/m1_training/
    """
    
    print(summary)
    
    # Save summary to file
    with open('/content/training_summary.txt', 'w') as f:
        f.write(summary)
    
    print("\n💾 Summary saved to training_summary.txt")

## Troubleshooting

### If Training Stops
- Colab sessions timeout after ~90 minutes of inactivity
- Training automatically saves checkpoints every epoch
- Re-run the training cell - it will resume from last checkpoint

### If Out of Memory
```python
# Edit train_m1.py and reduce batch size
config = {
    'batch_size': 64,  # Reduced from 128
    # ... rest stays same
}
```

### If GPU Not Available
- Go to: Runtime → Change runtime type → GPU
- Select T4, A100, or V100

### If Dataset Not Found
- Check paths in Step 3
- Verify ZIP file uploaded to correct Drive location
- Run: `!ls -la /content/dataset/`